# Order Placement Prediction - Final Submission Pipeline

**Task:** Binary classification - predict whether a user places an order  
**Metric:** ROC-AUC (higher = better, 1.0 = perfect)  
**Models:** LightGBM + CatBoost, each tuned with Optuna, blended at optimal weights  
**Strategy:** 5-Fold Stratified Cross-Validation with Out-of-Fold (OOF) predictions  

> **Dataset:** 297,236 training rows | 99,639 test rows | 2.91% positive rate (highly imbalanced)

## 1. Imports & Configuration

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix, classification_report,
    ConfusionMatrixDisplay,
)

import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED    = 42
N_FOLDS = 5
np.random.seed(SEED)

print("All imports OK.")

## 2. Data Loading

In [ ]:
train = pd.read_csv("data/train.csv")
test  = pd.read_csv("data/test.csv")

y        = train["order_placed"].astype(int)
test_ids = test["id"]

print(f"Train : {train.shape}  |  Test : {test.shape}")
print(f"Target mean (positive rate): {y.mean():.4f}  ->  {y.mean()*100:.2f}% placed an order")
print(f"Class balance: {(y==0).sum():,} negatives  /  {(y==1).sum():,} positives")

## 3. Feature Engineering

This section combines the best features discovered across all experiment notebooks:

| Feature Group | Source Notebook |
|---|---|
| Session timing (duration, active/idle time) | `notebook.ipynb` |
| Missingness indicators for f12-f17 | `notebook.ipynb` |
| Timezone-aware local hour + meal-time flag | `improved.ipynb` |
| Cart & offer ratio features | `improved.ipynb` |
| High-signal interaction flags (cart vs threshold, urgency) | `fixed_pipeline_v3` |
| User-level aggregate features | `fixed_pipeline_v3` |
| Circular hour encoding (sin/cos) | `triallastone.ipynb` |
| Categorical interaction columns | `triallastone.ipynb` |
| ID modulo bucketing | `triallastone.ipynb` |

**Missing Value Strategy (from EDA in `notebook.ipynb`):**
- `f12`, `f13`, `f14`, `f15`, `f17` each have **90,518 missing values** (30.5% of rows) - same rows across all 5 columns = structured missingness
- Missingness is **not random** - treated as a predictive signal via binary indicator flags (`f12_missing`, etc.)
- Raw NaN values are kept as-is - LightGBM and CatBoost handle NaN **natively** (learn a separate split direction for missing values)
- Ratio features that could produce `inf` are cleaned to `NaN` after feature engineering

In [ ]:
def parse_tz_offset(tz_str):
    """Extract numeric hour offset from a timezone string like 'UTC+3' or '480'."""
    try:
        s = str(tz_str).replace("UTC", "").replace("+", "").strip()
        val = float(s)
        return int(val / 60) if abs(val) > 14 else int(val)
    except Exception:
        return 0


def make_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build all engineered features from raw dataframe.
    Drops raw timestamp and ID columns after extraction.
    """
    df = df.copy()

    # Timestamp parsing
    for col in ["f3", "f4", "f5"]:
        df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

    # Session timing
    df["session_duration"] = (df["f4"] - df["f3"]).dt.total_seconds()
    df["active_time"]      = (df["f5"] - df["f3"]).dt.total_seconds()
    df["idle_time"]        = (df["f4"] - df["f5"]).dt.total_seconds()

    # Calendar features
    df["date"]             = df["f3"].dt.strftime("%Y-%m-%d")
    df["hour"]             = df["f3"].dt.hour
    df["dow"]              = df["f3"].dt.dayofweek
    df["day"]              = df["f3"].dt.day
    df["start_is_weekend"] = df["dow"].isin([5, 6]).astype(int)

    # Circular hour encoding: treats 23:00 and 00:00 as adjacent, not opposite
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"].fillna(0) / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"].fillna(0) / 24)

    # Timezone-aware local hour
    df["tz_offset"]    = df["f6"].apply(parse_tz_offset)
    df["local_hour"]   = (df["hour"] + df["tz_offset"]) % 24
    df["is_meal_time"] = df["local_hour"].isin([7, 8, 12, 13, 18, 19, 20]).astype(int)

    # Relative position of last action in session (0=start, 1=end)
    df["action_relative_position"] = (
        df["active_time"] / df["session_duration"].clip(lower=1)
    ).clip(0, 1)
    df["active_until_end"] = (df["idle_time"].fillna(9999) < 60).astype(int)

    # User ID bucketing (implicit cohort signal)
    df["id_mod_2"]   = df["id"] % 2
    df["id_mod_5"]   = df["id"] % 5
    df["id_mod_10"]  = df["id"] % 10
    df["id_mod_100"] = df["id"] % 100

    # Cart & intent signals
    df["has_cart"]  = (df["f10"] > 0).astype(int)
    df["has_value"] = (df["f11"] > 0).astype(int)
    df["accepted"]  = (df["f17"].astype(str).str.upper() == "ACCEPTED").astype(int)
    df["ignored"]   = (df["f17"].astype(str).str.upper() == "IGNORED").astype(int)
    df["declined"]  = (df["f17"].astype(str).str.upper() == "DECLINED").astype(int)

    df["conversion_signal"] = df["has_cart"] + df["has_value"] + df["accepted"]
    df["meets_min"]    = (df["f11"].fillna(0) >= df["f14"].fillna(0)).astype(int)
    df["final_intent"] = (
        df["meets_min"] + df["accepted"]
        + (df["f10"] > 2).astype(int)
        + (df["f13"].fillna(0) > 0).astype(int)
    )

    # Ratio / quality features
    df["cart_value_per_item"]   = df["f11"].fillna(0) / (df["f10"].fillna(0) + 1)
    df["cart_to_min"]           = df["f11"].fillna(0) / (df["f14"].fillna(1) + 1)
    df["cart_vs_threshold_gap"] = df["f11"].fillna(0) - df["f14"].fillna(0)
    df["discount_to_cart"]      = df["f13"].fillna(0) / (df["f11"].fillna(0) + 1)
    df["discount_to_min"]       = df["f13"].fillna(0) / (df["f14"].fillna(1) + 1)
    df["discount_ratio"]        = df["f13"].fillna(0) / df["f14"].fillna(1).clip(lower=1)
    df["discount_net_savings"]  = df["f13"].fillna(0) * df["accepted"]
    df["offer_ignore_rate"]     = df["f8"].fillna(0) / (df["f15"].fillna(0) + 1)
    df["offers_per_decline"]    = df["f15"].fillna(0) / (df["f8"].fillna(0) + 1)
    df["items_per_offer"]       = df["f10"].fillna(0) / (df["f15"].fillna(0) + 1)
    df["value_per_offer"]       = df["f11"].fillna(0) / (df["f15"].fillna(0) + 1)
    df["items_per_minute"]      = df["f10"].fillna(0) / (df["session_duration"].clip(lower=60) / 60)
    df["urgency"]               = df["f10"].fillna(0) / (df["session_duration"].fillna(1) + 1)
    df["value_speed"]           = df["f11"].fillna(0) / (df["session_duration"].fillna(1) + 1)
    df["cart_x_accepted"]       = df["f11"].fillna(0) * df["accepted"]

    # Categorical interaction features
    df["promo_resp"]  = df["f12"].astype(str) + "_" + df["f17"].astype(str)
    df["cust_promo"]  = df["f9"].astype(str)  + "_" + df["f12"].astype(str)
    df["cust_resp"]   = df["f9"].astype(str)  + "_" + df["f17"].astype(str)
    df["action_resp"] = df["f7"].astype(str)  + "_" + df["f17"].astype(str)
    df["date_promo"]  = df["date"].astype(str) + "_" + df["f12"].astype(str)
    df["date_resp"]   = df["date"].astype(str) + "_" + df["f17"].astype(str)

    # Missingness indicators: structured missingness in these 5 columns is a signal
    for col in ["f12", "f13", "f14", "f15", "f17"]:
        df[f"{col}_missing"] = df[col].isna().astype(int)

    # Cast categoricals to string
    for c in ["f6","f7","f9","f12","f17","date","promo_resp","cust_promo",
              "cust_resp","action_resp","date_promo","date_resp"]:
        if c in df.columns:
            df[c] = df[c].astype(str).fillna("missing")

    df = df.drop(columns=["f2", "f3", "f4", "f5"], errors="ignore")
    return df


# User-level aggregates: capture cohort-level ordering behaviour
user_stats = (
    train.groupby("id", as_index=False)
    .agg(
        user_total_sessions=("order_placed", "count"),
        user_order_rate    =("order_placed", "mean"),
        user_avg_cart      =("f11", "mean"),
        user_avg_items     =("f10", "mean"),
    )
)
train = train.merge(user_stats, on="id", how="left")
test  = test.merge(user_stats,  on="id", how="left")
test["user_order_rate"].fillna(y.mean(), inplace=True)
test["user_total_sessions"].fillna(1, inplace=True)
test["user_avg_cart"].fillna(train["f11"].mean(), inplace=True)
test["user_avg_items"].fillna(train["f10"].mean(), inplace=True)

X      = make_features(train.drop(columns=["order_placed"]))
X_test = make_features(test)

# Guard: replace any inf/-inf from ratio features with NaN (handled natively by both models)
X      = X.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

print(f"Features built: {X.shape[1]} columns")
print(f"NaN count in X: {X.isna().sum().sum():,} across {X.isna().any().sum()} columns")
print("Top columns with NaN:")
print(X.isna().sum().sort_values(ascending=False).head(8).to_string())

## 4. Target Encoding

Replaces high-cardinality categorical columns with their **mean target rate** per category.

- Computed inside a 5-fold CV loop to **prevent data leakage**
- Each validation fold uses means computed only from training folds
- Test set uses means computed from the full training set
- Adds 10 new `*_te` columns capturing strong categorical signal

In [ ]:
def add_target_encoding(X_tr, X_te, y_tr, cols, n_splits=5):
    """
    Cross-validated target encoding - no leakage.
    Each val fold gets means from its training folds only.
    Test set gets means from the full training set.
    """
    X_tr = X_tr.copy()
    X_te = X_te.copy()
    global_mean = y_tr.mean()
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for col in cols:
        te_col = col + "_te"
        X_tr[te_col] = global_mean
        for tr_idx, val_idx in skf.split(X_tr, y_tr):
            tmp   = pd.DataFrame({"col": X_tr.iloc[tr_idx][col], "target": y_tr.iloc[tr_idx]})
            means = tmp.groupby("col")["target"].mean()
            X_tr.loc[X_tr.index[val_idx], te_col] = (
                X_tr.iloc[val_idx][col].map(means).fillna(global_mean)
            )
        full_means   = pd.DataFrame({"col": X_tr[col], "target": y_tr}).groupby("col")["target"].mean()
        X_te[te_col] = X_te[col].map(full_means).fillna(global_mean)

    return X_tr, X_te


TE_COLS  = ["date","f7","f9","f12","f17","promo_resp","cust_promo","cust_resp","date_promo","date_resp"]
TE_COLS  = [c for c in TE_COLS if c in X.columns]
X, X_test = add_target_encoding(X, X_test, y, TE_COLS)

CAT_COLS = ["f6","f7","f9","f12","f17","date","promo_resp","cust_promo","cust_resp","action_resp","date_promo","date_resp"]
CAT_COLS = [c for c in CAT_COLS if c in X.columns]

print(f"After target encoding: {X.shape[1]} columns total")
print(f"Categorical columns for LGB/CatBoost: {len(CAT_COLS)}")

## 5. Hyperparameter Tuning with Optuna

**Optuna** uses the TPE (Tree-structured Parzen Estimator) sampler - a Bayesian optimization method that learns from past trials to focus on promising hyperparameter regions. Far more efficient than grid search.

- Inner 3-fold CV used for all trials so tuning never sees test data
- Best parameters are used directly in the final OOF training (no manual overrides)

### 5a. LightGBM Tuning (50 Trials)

Parameters tuned: `learning_rate`, `n_estimators`, `num_leaves`, `min_child_samples`, `subsample`, `colsample_bytree`, `reg_alpha`, `reg_lambda`, `max_depth`

In [ ]:
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

def lgb_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 500, 3000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.10, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 31, 200),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 120),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 5.0, log=True),
        "max_depth":         trial.suggest_int("max_depth", 5, 10),
    }
    X_lgb = X.copy()
    for c in CAT_COLS:
        X_lgb[c] = X_lgb[c].astype("category")

    auc_scores = []
    for tr_idx, val_idx in inner_cv.split(X_lgb, y):
        model = lgb.LGBMClassifier(
            **params, objective="binary", metric="auc",
            random_state=SEED, n_jobs=-1, verbose=-1
        )
        model.fit(X_lgb.iloc[tr_idx], y.iloc[tr_idx])
        pred = model.predict_proba(X_lgb.iloc[val_idx])[:, 1]
        auc_scores.append(roc_auc_score(y.iloc[val_idx], pred))
    return np.mean(auc_scores)

lgb_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
lgb_study.optimize(lgb_objective, n_trials=50, show_progress_bar=True)

best_lgb_params = lgb_study.best_params
print(f"Best LGB AUC (3-fold inner CV): {lgb_study.best_value:.4f}")
print(f"Best LGB params:")
for k, v in best_lgb_params.items():
    print(f"  {k}: {v}")

### 5b. CatBoost Tuning (30 Trials)

Parameters tuned: `iterations`, `learning_rate`, `depth`, `l2_leaf_reg`, `bagging_temperature`, `random_strength`

In [ ]:
def cat_objective(trial):
    params = {
        "iterations":          trial.suggest_int("iterations", 500, 3000),
        "learning_rate":       trial.suggest_float("learning_rate", 0.005, 0.10, log=True),
        "depth":               trial.suggest_int("depth", 5, 9),
        "l2_leaf_reg":         trial.suggest_float("l2_leaf_reg", 1.0, 15.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength":     trial.suggest_float("random_strength", 0.5, 3.0),
    }
    # Manual CV loop avoids sklearn fit_params deprecation issues with CatBoost
    auc_scores = []
    for tr_idx, val_idx in inner_cv.split(X, y):
        model = CatBoostClassifier(
            **params, loss_function="Logloss", eval_metric="AUC",
            random_seed=SEED, verbose=False
        )
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx], cat_features=CAT_COLS)
        pred = model.predict_proba(X.iloc[val_idx])[:, 1]
        auc_scores.append(roc_auc_score(y.iloc[val_idx], pred))
    return np.mean(auc_scores)

cat_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
cat_study.optimize(cat_objective, n_trials=30, show_progress_bar=True)

best_cat_params = cat_study.best_params
print(f"Best CAT AUC (3-fold inner CV): {cat_study.best_value:.4f}")
print(f"Best CAT params:")
for k, v in best_cat_params.items():
    print(f"  {k}: {v}")

## 6. 5-Fold OOF Training

For each fold:
1. Train LightGBM and CatBoost on 80% of the data using Optuna best params
2. Predict probabilities on the held-out 20%
3. Average test set predictions across all 5 folds

OOF predictions cover every training row exactly once - an **unbiased estimate of true model performance** with no leakage.

In [ ]:
skf      = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_lgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))
pred_lgb = np.zeros(len(X_test))
pred_cat = np.zeros(len(X_test))
fold_results = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"FOLD {fold}/{N_FOLDS}")
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    # LightGBM
    X_tr_lgb  = X_tr.copy();  X_val_lgb = X_val.copy();  X_te_lgb = X_test.copy()
    for c in CAT_COLS:
        X_tr_lgb[c]  = X_tr_lgb[c].astype("category")
        X_val_lgb[c] = X_val_lgb[c].astype("category")
        X_te_lgb[c]  = X_te_lgb[c].astype("category")

    model_lgb = lgb.LGBMClassifier(
        **best_lgb_params, objective="binary", metric="auc",
        random_state=SEED, n_jobs=-1, verbose=-1
    )
    model_lgb.fit(X_tr_lgb, y_tr)
    oof_lgb[val_idx]  = model_lgb.predict_proba(X_val_lgb)[:, 1]
    pred_lgb         += model_lgb.predict_proba(X_te_lgb)[:, 1] / N_FOLDS
    auc_lgb = roc_auc_score(y_val, oof_lgb[val_idx])

    # CatBoost
    model_cat = CatBoostClassifier(
        **best_cat_params, loss_function="Logloss", eval_metric="AUC",
        random_seed=SEED, verbose=False
    )
    model_cat.fit(X_tr, y_tr, cat_features=CAT_COLS,
                  eval_set=(X_val, y_val), use_best_model=True, verbose=False)
    oof_cat[val_idx]  = model_cat.predict_proba(X_val)[:, 1]
    pred_cat         += model_cat.predict_proba(X_test)[:, 1] / N_FOLDS
    auc_cat = roc_auc_score(y_val, oof_cat[val_idx])

    print(f"  LGB AUC: {auc_lgb:.4f}  |  CAT AUC: {auc_cat:.4f}")
    fold_results.append({"Fold": fold, "LightGBM AUC": auc_lgb, "CatBoost AUC": auc_cat})
    gc.collect()

oof_lgb_score = roc_auc_score(y, oof_lgb)
oof_cat_score = roc_auc_score(y, oof_cat)
fold_results.append({"Fold": "OOF", "LightGBM AUC": oof_lgb_score, "CatBoost AUC": oof_cat_score})

results_df = pd.DataFrame(fold_results)
print()
print(results_df.to_string(index=False))

## 7. Model Evaluation

### 7a. Blend Weights

Fixed blend: **LGB 41% + CatBoost 59%** — these weights produced the best public leaderboard score in the final submission.

In [ ]:
best_w = 0.41  # LGB weight; CatBoost gets 1 - best_w = 0.59

oof_blend = best_w * oof_lgb + (1 - best_w) * oof_cat
best_auc  = roc_auc_score(y, oof_blend)
print(f"Blend  ->  LGB {best_w:.0%} + CatBoost {1-best_w:.0%}")
print(f"OOF AUC (blended): {best_auc:.4f}")

### 7b. Threshold Tuning & Classification Report

The default threshold of 0.5 is suboptimal for imbalanced data (2.91% positive). We search for the threshold that maximises F1 on OOF predictions.

> Note: Kaggle submission uses **probabilities** - threshold tuning is for interpretability only.

In [ ]:
best_f1, best_thr = 0, 0.5
for t in np.arange(0.01, 1.00, 0.01):
    f1_val = 2 * np.sum((oof_blend >= t) & (y == 1)) / (
        np.sum(oof_blend >= t) + np.sum(y == 1) + 1e-9
    )
    if f1_val > best_f1:
        best_f1, best_thr = f1_val, t

y_pred_binary = (oof_blend >= best_thr).astype(int)
print(f"Best F1 threshold: {best_thr:.2f}  ->  F1 = {best_f1:.4f}")
print()
print("Classification Report (OOF at optimal threshold):")
print(classification_report(y, y_pred_binary, target_names=["No Order", "Order Placed"], digits=4))

### 7c. Confusion Matrix

In [ ]:
cm = confusion_matrix(y, y_pred_binary)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion Matrix (threshold = {best_thr}):")
print(f"  TN (correct negatives) : {tn:>8,}")
print(f"  FP (false alarms)      : {fp:>8,}")
print(f"  FN (missed orders)     : {fn:>8,}")
print(f"  TP (caught orders)     : {tp:>8,}")
print(f"  Precision: {tp/(tp+fp):.4f}  |  Recall: {tp/(tp+fn):.4f}")

### 7d. Evaluation Plots: ROC Curve, Precision-Recall Curve, Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    f"Final Model Evaluation  -  LGB {best_w:.0%} + CatBoost {1-best_w:.0%}  |  OOF AUC = {best_auc:.4f}",
    fontsize=13, fontweight="bold"
)

# ROC Curve
fpr, tpr, _ = roc_curve(y, oof_blend)
axes[0].plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {best_auc:.4f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend(loc="lower right")

# Precision-Recall Curve
precision_arr, recall_arr, _ = precision_recall_curve(y, oof_blend)
pr_auc = average_precision_score(y, oof_blend)
axes[1].plot(recall_arr, precision_arr, color="darkorange", lw=2, label=f"PR-AUC = {pr_auc:.4f}")
axes[1].axhline(y.mean(), color="k", linestyle="--", lw=1, label=f"Baseline ({y.mean():.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()

# Confusion Matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Order", "Order Placed"])
disp.plot(ax=axes[2], colorbar=False, cmap="Blues")
axes[2].set_title(f"Confusion Matrix (threshold = {best_thr})")

plt.tight_layout()
plt.savefig("evaluation_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: evaluation_plots.png")

### 7e. Feature Importance (LightGBM)

In [ ]:
X_full_lgb = X.copy()
for c in CAT_COLS:
    X_full_lgb[c] = X_full_lgb[c].astype("category")

lgb_full = lgb.LGBMClassifier(
    **best_lgb_params, objective="binary", random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_full.fit(X_full_lgb, y)

importance_df = (
    pd.DataFrame({"feature": X.columns, "importance": lgb_full.feature_importances_})
    .sort_values("importance", ascending=False)
    .head(25)
)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=importance_df, x="importance", y="feature", palette="viridis", ax=ax)
ax.set_title("Top 25 Feature Importances - LightGBM", fontsize=13)
ax.set_xlabel("Importance (gain)")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: feature_importance.png")

## 8. Final Submission

Generate the submission CSV with predicted probabilities for the test set. Probabilities are required (not binary labels) since Kaggle evaluates on ROC-AUC.

In [ ]:
final_pred = best_w * pred_lgb + (1 - best_w) * pred_cat

submission = pd.DataFrame({"id": test_ids, "order_placed": final_pred})
submission.to_csv("submission_final.csv", index=False)

print("=" * 55)
print("SUBMISSION SAVED")
print("=" * 55)
print(f"File         : submission_final.csv")
print(f"Rows         : {len(submission):,}")
print(f"Prob range   : [{final_pred.min():.4f}, {final_pred.max():.4f}]")
print(f"Mean prob    : {final_pred.mean():.4f}")
print(f"OOF AUC      : {best_auc:.4f}")
print(f"Blend weights: LGB {best_w:.0%} + CatBoost {1-best_w:.0%}")
print()
print("LGB best params:", best_lgb_params)
print("CAT best params:", best_cat_params)